# ODLS on fastMRI knee multicoil (CORPD_FBK) -- Colab runner

Assumes you've already downloaded the 20 files from
https://www.kaggle.com/datasets/arafatshovon/fastmri-knee-multicoil
and uploaded them to a folder in your Google Drive -- either as loose
`.h5` files or as a single `.zip`/`.tar` archive containing them.

This notebook:
1. Mounts your Drive.
2. Copies/extracts the 20 files to local Colab disk (much faster random
   access than reading directly off Drive during training).
3. Filters to the `CORPD_FBK` acquisition only (excludes `CORPDFS_FBK`).
4. Splits the resulting files into train/val folders.
5. Clones the `1D_MRI` GitHub repo and runs `train.py` against the
   fastMRI adapter (`fastmri_data.py`).

Edit the **Config** cell below to match your Drive folder path, then run
all cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config -- edit these paths

In [ ]:
# Folder in your Drive holding the 20 downloaded files (loose .h5 files
# and/or a single archive containing them -- both are handled below).
DRIVE_DATA_DIR = "/content/drive/MyDrive/fastmri_knee_corpd"

# Where to stage a local (fast-disk) copy on the Colab VM.
LOCAL_RAW_DIR = "/content/fastmri_raw"
LOCAL_TRAIN_DIR = "/content/fastmri_corpd/train"
LOCAL_VAL_DIR = "/content/fastmri_corpd/val"

REPO_URL = "https://github.com/Shambhawi419/1D_MRI.git"
REPO_DIR = "/content/1D_MRI"

VAL_FRACTION = 0.2       # fraction of CORPD_FBK files held out for validation
SPLIT_SEED = 0

N_COILS = 8               # must match N_VIRTUAL_COILS below
N_VIRTUAL_COILS = 8
MASK_TYPE = "cartesian"
AF = 4.0

## Stage the data locally

Copies everything from `DRIVE_DATA_DIR` into `LOCAL_RAW_DIR`, extracting
any `.zip`/`.tar*` archive found along the way, so `LOCAL_RAW_DIR` ends up
flat with `.h5` files regardless of how the 20 files were stored in Drive.

In [ ]:
import os
import shutil
import tarfile
import zipfile

os.makedirs(LOCAL_RAW_DIR, exist_ok=True)

if not os.path.isdir(DRIVE_DATA_DIR):
    raise FileNotFoundError(
        f"{DRIVE_DATA_DIR} not found -- check the Drive is mounted and the "
        "folder path/name is correct."
    )

n_copied, n_extracted = 0, 0
for name in sorted(os.listdir(DRIVE_DATA_DIR)):
    src = os.path.join(DRIVE_DATA_DIR, name)
    if os.path.isdir(src):
        continue

    if name.lower().endswith(".h5") or name.lower().endswith(".hdf5"):
        dst = os.path.join(LOCAL_RAW_DIR, name)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
        n_copied += 1

    elif zipfile.is_zipfile(src):
        with zipfile.ZipFile(src) as zf:
            zf.extractall(LOCAL_RAW_DIR)
        n_extracted += 1

    elif tarfile.is_tarfile(src):
        with tarfile.open(src) as tf:
            tf.extractall(LOCAL_RAW_DIR)
        n_extracted += 1

# Archives sometimes unpack into a nested subfolder -- flatten any .h5
# files found below LOCAL_RAW_DIR up to its top level.
for root, _, files in os.walk(LOCAL_RAW_DIR):
    if root == LOCAL_RAW_DIR:
        continue
    for f in files:
        if f.lower().endswith((".h5", ".hdf5")):
            src = os.path.join(root, f)
            dst = os.path.join(LOCAL_RAW_DIR, f)
            if not os.path.exists(dst):
                shutil.move(src, dst)

h5_files = sorted(
    f for f in os.listdir(LOCAL_RAW_DIR) if f.lower().endswith((".h5", ".hdf5"))
)
print(f"copied {n_copied} loose .h5 files, extracted {n_extracted} archive(s)")
print(f"{len(h5_files)} .h5 files staged locally in {LOCAL_RAW_DIR}")

## Clone the repo and filter to CORPD_FBK

In [ ]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "odls"))

from fastmri_data import find_corpd_files

corpd_files = find_corpd_files(LOCAL_RAW_DIR, fat_suppressed=False)
print(f"{len(corpd_files)} of {len(h5_files)} staged files are CORPD_FBK")
for p in corpd_files:
    print(" ", os.path.basename(p))

if not corpd_files:
    raise RuntimeError(
        "No CORPD_FBK files found among the staged data -- if your 20 "
        "files are the fat-suppressed variant instead, re-run with "
        "fat_suppressed=True, or use --fastmri-fat-suppressed in the "
        "training cell below."
    )

## Split into train/val folders

`train.py` / `evaluate.py` take a directory each, so the CORPD_FBK files
found above are split and symlinked into two separate folders.

In [ ]:
import random

os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)
os.makedirs(LOCAL_VAL_DIR, exist_ok=True)

rng = random.Random(SPLIT_SEED)
shuffled = corpd_files[:]
rng.shuffle(shuffled)

n_val = max(1, int(round(len(shuffled) * VAL_FRACTION))) if len(shuffled) > 1 else 0
val_files = shuffled[:n_val]
train_files = shuffled[n_val:]

def _relink(files, dest_dir):
    for src in files:
        link_path = os.path.join(dest_dir, os.path.basename(src))
        if os.path.lexists(link_path):
            os.remove(link_path)
        os.symlink(src, link_path)

_relink(train_files, LOCAL_TRAIN_DIR)
_relink(val_files, LOCAL_VAL_DIR)

print(f"train: {len(train_files)} files -> {LOCAL_TRAIN_DIR}")
print(f"val:   {len(val_files)} files -> {LOCAL_VAL_DIR}")

## Train

In [ ]:
!cd {REPO_DIR}/odls && python train.py \
    --fastmri-train-root {LOCAL_TRAIN_DIR} \
    --fastmri-val-root {LOCAL_VAL_DIR} \
    --n-coils {N_COILS} --n-virtual-coils {N_VIRTUAL_COILS} \
    --mask-type {MASK_TYPE} --af {AF} \
    --checkpoint-dir /content/checkpoints

## Evaluate

Reuses the val split as a stand-in test set here since only 20 files are
available in total; swap in a separate held-out folder if/when you have
more data.

In [ ]:
!cd {REPO_DIR}/odls && python evaluate.py \
    --fastmri-test-root {LOCAL_VAL_DIR} \
    --checkpoint /content/checkpoints/odls_best.pt \
    --n-coils {N_COILS} --n-virtual-coils {N_VIRTUAL_COILS} \
    --mask-type {MASK_TYPE} --af {AF}